# ONDO 신발·가방 착용 생성 (OmniTry · FLUX Fill nf4 · Kaggle T4)
신발·가방은 CatVTON이 학습하지 않은 물건이라 OmniTry(신발·가방 지원 가상 착용 모델)로 만듭니다. 발/상체 부분만 잘라서 생성한 뒤 원래 모델 사진에 다시 붙입니다.

In [ ]:
# ===== 설정 =====
PRIORITY_MAX = 1
CATEGORIES = ["shoes", "bag"]
TEST_IDS = None
MODES = ["fp16"]             # GPU별 연산 정밀도 (테스트에서는 fp16/bf16 비교)
SAME_JOBS_ALL_GPUS = False
STEPS = 20
GUIDANCE = 30
SEED = 42
W, H = 768, 1024
MAX_AREA = 640 * 640          # 잘라낸 부분을 이 크기로 생성
TIME_BUDGET_MIN = 300
STALL_MIN = 15
SAVE_DEBUG = False
OUT_NAME = "results"


In [ ]:
import subprocess, sys, os, importlib.metadata as md, time
t=time.time()
if not os.path.exists("/tmp/OmniTry"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kunbyte-AI/OmniTry.git", "/tmp/OmniTry"], check=True)
pins = []
for p in ["numpy", "numba", "torch", "torchvision"]:
    try: pins.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: pass
open("/tmp/constraints.txt", "w").write("\n".join(pins) + "\n"); print("고정:", pins)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt",
    "diffusers==0.33.1", "transformers==4.45.0", "huggingface_hub==0.30.2", "accelerate==1.1.1", "peft==0.13.2",
    "bitsandbytes==0.45.5", "sentencepiece", "protobuf", "omegaconf", "einops", "rembg==2.0.59", "onnxruntime"], check=True)
for p in ["torch", "diffusers", "transformers", "peft", "bitsandbytes"]: print(p, md.version(p))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import snapshot_download
BASE_DIR = snapshot_download("camenduru/FLUX.1-Fill-dev-ungated", local_dir="/tmp/flux_fill_base",
    allow_patterns=["model_index.json", "scheduler/*", "vae/*", "text_encoder/*", "text_encoder_2/*", "tokenizer/*", "tokenizer_2/*"])
snapshot_download("diffusers/FLUX.1-Fill-dev-nf4", allow_patterns=["transformer/*"])
LORA_DIR = snapshot_download("Kunbyte/OmniTry", local_dir="/tmp/omnitry_ckpt", allow_patterns=["omnitry_v1_unified.safetensors"])
print("다운로드 완료", int(time.time()-t), "s", flush=True)


In [ ]:
# 문구 임베딩은 신발/가방 2개뿐이라 CPU에서 한 번만 계산 (T5 fp16 오버플로 방지)
import torch, time, gc
t=time.time()
from diffusers import FluxFillPipeline as _P
tp = _P.from_pretrained(BASE_DIR, transformer=None, vae=None, torch_dtype=torch.bfloat16)
EMB = {}
with torch.no_grad():
    for cls, text in {"shoe": "replacing the shoe", "bag": "trying on bag"}.items():
        pe, pp, _ = tp.encode_prompt(prompt=text, prompt_2=None, device="cpu", max_sequence_length=512)
        EMB[cls] = (pe.float(), pp.float()); print(cls, tuple(pe.shape), tuple(pp.shape), float(pe.abs().max()))
torch.save(EMB, "/tmp/omni_embeds.pt"); del tp; gc.collect(); print("임베딩 완료", int(time.time()-t), "s")


In [ ]:
WORKER_SRC = "import os, sys, csv, glob, time, json, threading, faulthandler, math\nos.environ.update(HF_HUB_DISABLE_PROGRESS_BARS=\"1\", TRANSFORMERS_VERBOSITY=\"error\", DIFFUSERS_VERBOSITY=\"error\", TQDM_DISABLE=\"1\")\nSHARD, NSHARD = int(sys.argv[1]), int(sys.argv[2])\ncfg = json.load(open(\"/kaggle/working/omni_config.json\")); globals().update(cfg)\nMODE = MODES[SHARD % len(MODES)]           # 테스트: GPU별로 fp16 / bf16 비교\nT0 = time.time()\ndef log(*a): print(f\"[gpu{SHARD} {MODE} {time.time()-T0:6.0f}s]\", *a, flush=True)\n_beat = [time.time(), \"시작\"]\ndef beat(s): _beat[0] = time.time(); _beat[1] = s\ndef _watch():\n    while True:\n        time.sleep(30)\n        if time.time() - _beat[0] > STALL_MIN * 60:\n            log(f\"!!! '{_beat[1]}' 단계에서 {STALL_MIN}분 넘게 멈춤 → 종료\"); faulthandler.dump_traceback(all_threads=True); sys.stdout.flush(); os._exit(3)\nthreading.Thread(target=_watch, daemon=True).start()\nsys.path.insert(0, \"/tmp/OmniTry\")\nkit = os.path.dirname(glob.glob(\"/kaggle/input/**/jobs.csv\", recursive=True)[0])\nOUT = f\"/kaggle/working/{OUT_NAME}_{MODE}\" if len(MODES) > 1 else f\"/kaggle/working/{OUT_NAME}\"; os.makedirs(OUT, exist_ok=True)\ndone = set()\nfor p in glob.glob(\"/kaggle/input/**/results*/*/*.png\", recursive=True) + glob.glob(OUT + \"/*/*.png\"):\n    done.add(os.path.basename(os.path.dirname(p)) + \"_\" + os.path.splitext(os.path.basename(p))[0])\njobs = [r for r in csv.DictReader(open(f\"{kit}/jobs.csv\", encoding=\"utf-8\"))\n        if int(r[\"priority\"]) <= PRIORITY_MAX and r[\"category\"] in CATEGORIES]\nif TEST_IDS: jobs = [r for r in jobs if r[\"job_id\"] in TEST_IDS]\ntodo = [r for r in jobs if r[\"job_id\"] not in done]\nORDER = [\"wave_cool\", \"straight_warm\", \"straight_cool\", \"wave_warm\", \"natural_warm\", \"natural_cool\"]\ntodo.sort(key=lambda r: ORDER.index(r[\"body\"] + \"_\" + r[\"tone\"]))\ntodo = todo if SAME_JOBS_ALL_GPUS else todo[SHARD::NSHARD]\nlog(f\"대상 {len(jobs)}장 / 이 GPU가 생성 {len(todo)}장\")\nif not todo: sys.exit(0)\n\nimport torch, numpy as np, peft\nfrom PIL import Image, ImageFilter\nimport torchvision.transforms as T\nfrom omnitry.models.transformer_flux import FluxTransformer2DModel\nfrom omnitry.pipelines.pipeline_flux_fill import FluxFillPipeline\nfrom safetensors import safe_open\nDT = torch.float16 if MODE == \"fp16\" else torch.bfloat16\nbeat(\"transformer 로딩\"); log(\"transformer(nf4) 로딩\")\ntransformer = FluxTransformer2DModel.from_pretrained(\"diffusers/FLUX.1-Fill-dev-nf4\", subfolder=\"transformer\", torch_dtype=DT).requires_grad_(False)\nimport bitsandbytes as bnb\nfor m in transformer.modules():\n    if isinstance(m, bnb.nn.Linear4bit): m.compute_dtype = DT\nbeat(\"LoRA 적용\"); log(\"OmniTry LoRA 적용\")\nfrom peft import LoraConfig\nlcfg = LoraConfig(r=16, lora_alpha=16, init_lora_weights=\"gaussian\", target_modules=[\n    'x_embedder', 'attn.to_k', 'attn.to_q', 'attn.to_v', 'attn.to_out.0', 'attn.add_k_proj', 'attn.add_q_proj', 'attn.add_v_proj',\n    'attn.to_add_out', 'ff.net.0.proj', 'ff.net.2', 'ff_context.net.0.proj', 'ff_context.net.2', 'norm1_context.linear', 'norm1.linear',\n    'norm.linear', 'proj_mlp', 'proj_out'])\ntransformer.add_adapter(lcfg, adapter_name=\"vtryon_lora\"); transformer.add_adapter(lcfg, adapter_name=\"garment_lora\")\nwith safe_open(LORA_PATH, framework=\"pt\") as f:\n    sd = {k: f.get_tensor(k) for k in f.keys()}\nown = dict(transformer.named_parameters())\nhit = [k for k in sd if k in own]; log(f\"LoRA 키 {len(sd)}개 중 {len(hit)}개 일치\")\nif len(hit) < len(sd) * 0.95: log(\"!!! LoRA 키 불일치 예시:\", [k for k in sd if k not in own][:3], [k for k in own if 'lora' in k][:3])\nwith torch.no_grad():\n    for k in hit: own[k].data = sd[k].to(device=own[k].device, dtype=DT)\n    for n, p in transformer.named_parameters():\n        if \"lora_\" in n and p.dtype != DT: p.data = p.data.to(DT)\nfor n, p in transformer.named_parameters():\n    if \"lora_\" in n and p.device.type != \"cuda\": p.data = p.data.to(\"cuda\")\ndef make_fwd(module):\n    def one(self, name, x, *a, **kw):\n        r = self.base_layer(x, *a, **kw)\n        A, B = self.lora_A[name], self.lora_B[name]\n        return r + (B(A(x.to(A.weight.dtype))) * self.scaling[name]).to(r.dtype)\n    def fwd(self, x, *a, **kw):\n        return torch.cat((one(self, \"vtryon_lora\", x[:1], *a, **kw), one(self, \"garment_lora\", x[1:], *a, **kw)), dim=0)\n    return fwd.__get__(module, type(module))\nnh = 0\nfor n, m in transformer.named_modules():\n    if isinstance(m, peft.tuners.lora.layer.LoraLayer) and hasattr(m, \"base_layer\") and \"vtryon_lora\" in getattr(m, \"lora_A\", {}):\n        m.forward = make_fwd(m); nh += 1\nlog(f\"LoRA 분기 레이어 {nh}개\")\nif DT == torch.float16:\n    def _clip(o):\n        if torch.is_tensor(o): return torch.nan_to_num(o, nan=0.0, posinf=6.0e4, neginf=-6.0e4).clamp(-6.0e4, 6.0e4)\n        if isinstance(o, tuple): return tuple(_clip(x) for x in o)\n        return o\n    nb_ = 0\n    for blk in list(transformer.transformer_blocks) + list(transformer.single_transformer_blocks):\n        blk.register_forward_hook(lambda m, i, o: _clip(o)); nb_ += 1\n    log(f\"fp16 오버플로 방지 훅 {nb_}개\")\nbeat(\"파이프라인 로딩\"); log(\"VAE/스케줄러 로딩\")\npipe = FluxFillPipeline.from_pretrained(BASE_DIR, transformer=transformer, text_encoder=None, text_encoder_2=None, tokenizer=None, tokenizer_2=None, torch_dtype=DT)\npipe.vae.to(\"cuda\")\nEMB = torch.load(\"/tmp/omni_embeds.pt\")\nlog(\"모델 로딩 완료\")\n\ndef on_white(im):\n    im = im.convert(\"RGBA\"); bg = Image.new(\"RGBA\", im.size, (255, 255, 255, 255)); bg.alpha_composite(im); return bg.convert(\"RGB\")\n_av = {}\ndef avatar(name):\n    if name not in _av:\n        rgba = Image.open(f\"{kit}/avatars/{name}\").convert(\"RGBA\").resize((W, H), Image.LANCZOS)\n        _av[name] = (rgba, on_white(rgba))\n    return _av[name]\nLONG_BOOTS = [\"롱부츠\", \"니하이\", \"롱 부츠\", \"부츠\"]\ndef crop_box(job, rgba):\n    a = np.array(rgba.split()[-1]) > 40; ys, xs = np.where(a); top, bot = ys.min(), ys.max(); h = bot - top\n    if job[\"category\"] == \"shoes\":\n        long = any(k in job[\"subtype\"] + job[\"name\"] for k in LONG_BOOTS) and \"앵클\" not in job[\"subtype\"] + job[\"name\"]\n        y0 = top + int(h * (.56 if long else .66)); y1 = min(H, bot + int(h * .05))\n    else:   # 가방: 한쪽 손·어깨만 보이게 잘라서 가방이 하나만 생기도록\n        y0 = top + int(h * .08); y1 = top + int(h * .70)\n        cx = (xs.min() + xs.max()) // 2\n        x0 = max(0, xs.min() - int(h * .16)); x1 = min(W, cx + int(h * .04))\n        return x0, y0, x1, y1\n    ch = y1 - y0; cw = max(int(ch * .85), int((xs.max() - xs.min()) * 1.35))\n    cx = (xs.min() + xs.max()) // 2; x0 = max(0, cx - cw // 2); x1 = min(W, x0 + cw); x0 = max(0, x1 - cw)\n    return x0, y0, x1, y1\ndef feather(size, px=28):\n    w, h = size; m = np.ones((h, w), np.float32)\n    r = np.linspace(0, 1, px)\n    m[:px] *= r[:, None]; m[-px:] *= r[::-1][:, None]; m[:, :px] *= r[None]; m[:, -px:] *= r[::-1][None]\n    return Image.fromarray((m * 255).astype(np.uint8))\ndef run(job):\n    rgba, person = avatar(job[\"base_avatar\"])\n    box = crop_box(job, rgba); crop = person.crop(box)\n    s = math.sqrt(MAX_AREA / (crop.width * crop.height)); tW, tH = int(crop.width * s) // 16 * 16, int(crop.height * s) // 16 * 16\n    pt = T.Compose([T.Resize((tH, tW)), T.ToTensor()])(crop)\n    obj = on_white(Image.open(f\"{kit}/{job['image_path']}\"))\n    r = min(tW / obj.width, tH / obj.height) * .9\n    ot = T.Compose([T.Resize((int(obj.height * r), int(obj.width * r))), T.ToTensor()])(obj)\n    pad = torch.ones_like(pt); nh_, nw_ = ot.shape[1:]; y, x = (tH - nh_) // 2, (tW - nw_) // 2; pad[:, y:y + nh_, x:x + nw_] = ot\n    cls = \"shoe\" if job[\"category\"] == \"shoes\" else \"bag\"\n    pe, pp = EMB[cls]\n    cond = torch.stack([pt, pad]).to(\"cuda\", DT)\n    with torch.no_grad():\n        out = pipe(prompt_embeds=pe.to(\"cuda\", DT).repeat(2, 1, 1), pooled_prompt_embeds=pp.to(\"cuda\", DT).repeat(2, 1), height=tH, width=tW,\n                   img_cond=cond, mask=torch.zeros_like(cond), guidance_scale=GUIDANCE, num_inference_steps=STEPS,\n                   generator=torch.Generator(\"cuda\").manual_seed(SEED)).images[0]\n    arr = np.asarray(out).astype(np.float32)\n    if not np.isfinite(arr).all() or arr.mean() < 3: raise RuntimeError(\"검은 이미지/NaN (정밀도 문제)\")\n    gen = out.resize(crop.size, Image.LANCZOS)\n    full = person.copy(); full.paste(gen, box[:2], feather(crop.size))\n    # 배경 제거: 원래 모델 윤곽 + 새로 생긴 물건(가방·신발) 영역을 합친다 (rembg는 든 가방을 지워버림)\n    alpha = np.array(rgba.split()[-1]).astype(np.uint8)\n    g = np.asarray(gen.convert(\"RGB\")).astype(np.int16); c0 = np.asarray(crop.convert(\"RGB\")).astype(np.int16)\n    changed = (np.abs(g - c0).max(-1) > 30) & (g.min(-1) < 238)\n    cm = Image.fromarray((changed * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(5)).filter(ImageFilter.MinFilter(5)).filter(ImageFilter.MinFilter(3)).filter(ImageFilter.MaxFilter(3))\n    x0, y0, x1, y1 = box\n    region = alpha[y0:y1, x0:x1]; region = np.maximum(region, np.array(cm)); alpha[y0:y1, x0:x1] = region\n    a_img = Image.fromarray(alpha).filter(ImageFilter.GaussianBlur(0.8))\n    cut = full.convert(\"RGBA\"); cut.putalpha(a_img)\n    return cut, gen, crop\nmf_path = f\"{OUT}/manifest_{SHARD}.csv\"; fields = [\"job_id\",\"body\",\"tone\",\"variant_id\",\"product_id\",\"category\",\"subtype\",\"name\",\"color\",\"file\"]\nnewf = not os.path.exists(mf_path); mf = open(mf_path, \"a\", newline=\"\", encoding=\"utf-8\"); mw = csv.DictWriter(mf, fieldnames=fields)\nif newf: mw.writeheader()\ndeadline = T0 + TIME_BUDGET_MIN * 60; t0 = time.time()\nfor i, job in enumerate(todo, 1):\n    if time.time() > deadline: log(\"시간 예산 도달 → 저장 후 종료\"); break\n    beat(job[\"job_id\"]); ts = time.time()\n    try:\n        cut, gen, crop = run(job)\n        folder = f\"{OUT}/{job['body']}_{job['tone']}\"; os.makedirs(folder, exist_ok=True)\n        path = f\"{folder}/{job['variant_id']}.png\"; cut.save(path, optimize=True)\n        if SAVE_DEBUG:\n            os.makedirs(f\"{OUT}/_debug\", exist_ok=True)\n            dbg = Image.new(\"RGB\", (crop.width * 2, crop.height), \"white\"); dbg.paste(crop, (0, 0)); dbg.paste(gen, (crop.width, 0)); dbg.save(f\"{OUT}/_debug/{job['job_id']}.jpg\", quality=85)\n        mw.writerow({k: job[k] for k in fields if k != \"file\"} | {\"file\": os.path.relpath(path, OUT)}); mf.flush()\n        log(f\"{job['category']} {job['name']} 완료 {time.time()-ts:.1f}s\")\n    except Exception as e:\n        import traceback; log(\"실패\", job[\"job_id\"], repr(e)); traceback.print_exc(); sys.stdout.flush()\n    if i % 10 == 0: el = time.time() - t0; log(f\"{i}/{len(todo)}  {el/i:.1f}s/장  남은 예상 {el/i*(len(todo)-i)/60:.0f}분\")\nmf.close(); log(\"끝\")\n"


In [ ]:
import json, subprocess, torch, time
json.dump(dict(PRIORITY_MAX=PRIORITY_MAX, CATEGORIES=CATEGORIES, TEST_IDS=TEST_IDS, MODES=MODES, SAME_JOBS_ALL_GPUS=SAME_JOBS_ALL_GPUS,
    STEPS=STEPS, GUIDANCE=GUIDANCE, SEED=SEED, W=W, H=H, MAX_AREA=MAX_AREA, TIME_BUDGET_MIN=TIME_BUDGET_MIN, STALL_MIN=STALL_MIN,
    SAVE_DEBUG=SAVE_DEBUG, OUT_NAME=OUT_NAME, BASE_DIR=BASE_DIR, LORA_PATH=LORA_DIR + "/omnitry_v1_unified.safetensors"),
    open("/kaggle/working/omni_config.json", "w"))
open("/tmp/omni_worker.py", "w", encoding="utf-8").write(WORKER_SRC)
n = max(1, torch.cuda.device_count()); print("GPU", n, "장", flush=True)
procs = [subprocess.Popen([sys.executable, "-u", "/tmp/omni_worker.py", str(i), str(n)], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}) for i in range(n)]
hard = time.time() + (TIME_BUDGET_MIN + 20) * 60
while any(p.poll() is None for p in procs) and time.time() < hard: time.sleep(20)
for p in procs:
    if p.poll() is None: p.kill(); print("강제 종료한 작업 있음", flush=True)
print("종료 코드", [p.wait() for p in procs], flush=True)


In [ ]:
import glob, csv, os, zipfile
from PIL import Image
for OUT in sorted(glob.glob("/kaggle/working/results*")):
    rows = []
    for p in sorted(glob.glob(f"{OUT}/manifest_*.csv")): rows += list(csv.DictReader(open(p, encoding="utf-8")))
    if not rows: continue
    with open(f"{OUT}/manifest.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print(OUT, "생성", len(rows), "장")
with zipfile.ZipFile("/kaggle/working/ondo_accessory_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for OUT in glob.glob("/kaggle/working/results*"):
        for root, _, files in os.walk(OUT):
            for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "/kaggle/working"))
print("완료")
